# Notebook 3 — Adding sectors & shocks: the AI scenarios

This is where input-output analysis becomes a **policy/foresight tool**. Starting
from the compact, real-data model we built and saved in Notebook 2 (7 sectors ×
6 macro-regions), we will:

- **A.** run a simple, fully transparent **final-demand shock** on *Electrical
  equipment* — the "hello world" of scenario analysis;
- **B.** **add a new sector** that does not exist in official statistics yet —
  *Hyperscalers AI*, located only in the USA, with a GPU- and electricity-hungry
  cost structure, selling a *token-generation service*;
- **C.** build the headline scenario: **European service firms replace part of
  their high-skill labour with AI tokens**;
- **D.** read off the **winners and losers** with a few visualizations.

> 🧩 Everything downstream of "real EXIOBASE numbers" is illustrative: the AI cost
> structure and the substitution intensity are **plausible assumptions you can
> calibrate**, not official figures. The point is the *method*.

In [ ]:
import mario
import os
import numpy as np
import pandas as pd

mario.set_log_verbosity("warning")
print("MARIO version:", mario.__version__)

# Path to the parquet snapshot written by Notebook 2.
PARQUET_PATH = os.path.join("outputs", "exio_aggregated_parquet")
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

## Load the model saved by Notebook 2

We round-trip the parquet snapshot instead of re-parsing the multi-GB raw
EXIOBASE — fast and reproducible.

In [ ]:
db = mario.parse_from_parquet(
    PARQUET_PATH,
    table="IOT",
    mode="flows",
    new_scenario="baseline",
    flat=True,
)
db.calc_all(["X", "Z", "z", "Y", "v", "e"])
print(db)
print("\nSectors:", db.sectors)
print("Regions:", db.regions)
print("Value-added factors:", db.get_index("Factor of production"))
print("Satellite accounts:", db.get_index("Satellite account"))

In [ ]:
# Helper used throughout the notebook: total output X for one scenario, as a
# tidy Series indexed by (Region, Sector).
def output_series(database, scenario):
    x = database.query("X", scenario).copy()
    x.index = x.index.droplevel("Level")     # drop the 'Sector'/'Level' middle level
    return x.iloc[:, 0]

baseline_X = output_series(db, "baseline")
baseline_X.head(14)

## Part A — A simple final-demand shock

**Question:** what happens to the whole system if European households and firms
demand **+30% of Electrical equipment**?

Because we kept *Electrical equipment* as an explicit sector in Notebook 2, we can
shock it directly. MARIO's shock workflow is:

1. generate a shock template with `get_shock_excel(...)`;
2. fill the relevant sheet — here **`Y`** (final demand) — with the rows to shock;
3. apply it with `shock_calc(...)`, which creates a **new scenario** and leaves
   `baseline` untouched.

The `Y` sheet columns are `Region_from, Sector_from, Region_to,
Consumption category_to, type, value`. With `type="Percentage"`, MARIO applies
`Y_new = Y_old × (1 + value)`, so `value=0.30` means +30%.

In [ ]:
SHOCK_PATH = os.path.join(OUT_DIR, "shock_ee.xlsx")
db.get_shock_excel(path=SHOCK_PATH, num_shock=1)
print("Empty shock template written to:", SHOCK_PATH)

# Reference for filling: the exact label to use as the demand category
print("Consumption categories in the DB:", db.get_index("Consumption category"))

**Fill in the shock.** Open `outputs/shock_ee.xlsx`, go to the **`Y`** sheet
and write a single row (use the consumption-category label printed above):

| Region_from | Sector_from | Region_to | Consumption category_to | type | value |
|---|---|---|---|---|---|
| Europe | Electrical equipment | Europe | *(your final-demand category)* | Percentage | 0.30 |

Save the file, then apply it below.

In [ ]:
db.shock_calc(SHOCK_PATH, Y=True, scenario="ee_demand")
print("Scenarios now available:", db.scenarios)

### Read the effect

Compare total output by sector, baseline vs shock. The increase propagates **up
the supply chain**: not only Electrical equipment, but also the electricity, rare
materials and manufacturing that feed into it.

In [ ]:
ee_X = output_series(db, "ee_demand")
delta = (ee_X - baseline_X)
pct = (delta / baseline_X.replace(0, np.nan) * 100).rename("pct_change")

# Output change aggregated by sector (sum across regions), most affected first
by_sector = (
    pd.DataFrame({"baseline": baseline_X, "ee_demand": ee_X})
    .groupby(level="Item").sum()
)
by_sector["pct_change"] = (by_sector["ee_demand"] / by_sector["baseline"] - 1) * 100
by_sector.sort_values("pct_change", ascending=False)

In [ ]:
# Visualize the % output change by sector
plot_df = by_sector.reset_index().rename(columns={"Item": "Sector"})
fig = db.plot(
    data=plot_df,
    kind="bar",
    preset=None,
    x="Sector",
    y="pct_change",
    title="Output change from a +30% EU demand for Electrical equipment",
    path=False,
    auto_open=False,
)
fig

## Part B — Add a new sector: *Hyperscalers AI*

Official statistics don't have an "AI compute" sector yet. With `add_sectors(...)`
we can **inject one** by describing its cost structure (its column of the
input-output table) plus its total output and where that output goes.

We reload a **fresh** baseline so this structural change is independent of Part A.

In [ ]:
db = mario.parse_from_parquet(
    PARQUET_PATH, table="IOT", mode="flows", new_scenario="baseline", flat=True,
)
db.calc_all(["X", "z", "v", "e", "Y"])
baseline_X = output_series(db, "baseline")

### B.1 The cost structure of an AI hyperscaler

We describe *Hyperscalers AI* (USA) as an **inventory**: how much of each input it
needs to produce **one unit (1 M€) of token service**. In a monetary IOT the
intermediate-input shares plus the value-added share must sum to **1** (output =
costs + value added). Our assumptions:

| Input | Source region | Share of 1 M€ output |
|---|---|---|
| Electrical equipment (servers, GPUs) | USA | 0.18 |
| Electrical equipment (chips) | Rest of Asia | 0.12 |
| Non-renewable electricity | USA | 0.10 |
| Renewable electricity | USA | 0.08 |
| Rare materials | China | 0.04 |
| Manufacturing (cooling, construction) | USA | 0.05 |
| Services (real estate, telecom, O&M) | USA | 0.13 |
| **Value added** (capital + labour) | USA | **0.30** |
| **Total** | | **1.00** |

On top of the monetary structure we attach **direct environmental flows**: CO₂
(backup/own generation) and a large **Water** demand for cooling. These are
satellite accounts, so they sit *outside* the monetary balance. Their levels here
are illustrative placeholders — calibrate them to your favourite source.

### B.2 Size the new sector and the token price

We give the sector a baseline output level and a **token price** so we can talk in
tokens rather than euros. With a price of **$10 per 1 million tokens**, a
50 000 M€ (~\$54 bn) sector corresponds to a few thousand *trillion* tokens a
year — the right order of magnitude for today's frontier hyperscalers.

In the baseline, the whole output is sold to final demand (no sector buys AI
tokens yet — that is exactly what changes in Part C).

In [ ]:
AI_SECTOR = "Hyperscalers AI"
AI_REGION = "USA"
AI_OUTPUT = 50_000.0          # M€ of token service in the baseline

# Token economics (illustrative)
PRICE_PER_MILLION_TOKENS_USD = 10.0
EUR_USD = 1.08
million_tokens = (AI_OUTPUT * 1e6 * EUR_USD) / PRICE_PER_MILLION_TOKENS_USD
print(f"Implied volume: {million_tokens/1e6:,.0f} trillion tokens per year "
      f"at ${PRICE_PER_MILLION_TOKENS_USD}/1M tokens.\n")

# --- Reference values you need when filling the workbook (copy these strings) ---
SECTOR_UNIT = db.units["Sector"].iloc[0, 0]
SAT = db.units["Satellite account"]["unit"].to_dict()
VA_FACTORS = list(db.get_index("Factor of production"))
VA_LABEL = next((f for f in VA_FACTORS
                 if any(k in f.lower() for k in ["capital", "surplus", "operating"])),
                VA_FACTORS[0])
LABOUR_LABEL = next((f for f in VA_FACTORS
                     if any(k in f.lower() for k in
                            ["compensation", "labour", "labor", "wage", "employ"])),
                    VA_FACTORS[0])
CONS_CAT = db.get_index("Consumption category")[0]
print("Sector unit              :", SECTOR_UNIT)
print("Satellite units          :", SAT)
print("Value-added factors      :", VA_FACTORS)
print("  -> capital factor to use:", VA_LABEL)
print("  -> labour factor (Part C):", LABOUR_LABEL)
print("Consumption category     :", CONS_CAT)

### B.3 Generate the template and fill the recipe

`get_add_sectors_excel(...)` pre-fills the `Region`, `Sector`, `Inventory sheet`
and `Add or Split` columns for us; we complete the rest by hand.

In [ ]:
ADD_PATH = os.path.join(OUT_DIR, "add_ai_sector.xlsx")
db.get_add_sectors_excel(path=ADD_PATH, items=[AI_SECTOR], regions=[AI_REGION],
                         overwrite=True)
print("Empty add-sectors workbook written to:", ADD_PATH)

Open `outputs/add_ai_sector.xlsx` and fill two sheets.

**`Master` sheet** — complete the (already started) row for *Hyperscalers AI* /
*USA*:

| Quantity | Unit | Final consumption | Consumption category |
|---|---|---|---|
| 50000 | *(Sector unit, e.g. `M EUR`)* | 50000 | *(your consumption category)* |

(`Region`, `Sector`, `Inventory sheet` = `INV_001`, and `Add or Split` = `Add` are
pre-filled.)

**`INV_001` sheet** — the production recipe: how much of each input per **1 unit
(1 M€)** of token service. The `Sector` + `Factor of production` quantities are
shares that **sum to 1** (output = costs + value added). The two satellite rows are
direct environmental flows (use the satellite units printed above; the magnitudes
are illustrative placeholders to calibrate):

| Quantity | Unit | Input | Item type | DB Item | DB Region | Change type |
|---|---|---|---|---|---|---|
| 0.18 | M EUR | GPU/servers | Sector | Electrical equipment | USA | Update |
| 0.12 | M EUR | AI chips | Sector | Electrical equipment | Rest of Asia | Update |
| 0.10 | M EUR | grid power | Sector | Non-renewable electricity | USA | Update |
| 0.08 | M EUR | green power | Sector | Renewable electricity | USA | Update |
| 0.04 | M EUR | rare metals | Sector | Rare materials | China | Update |
| 0.05 | M EUR | cooling/build | Sector | Manufacturing | USA | Update |
| 0.13 | M EUR | real estate/O&M | Sector | Services | USA | Update |
| 0.30 | M EUR | capital+labour | Factor of production | *(your capital factor)* | USA | Update |
| 50000 | *(CO2 unit)* | direct CO2 | Satellite account | CO2 | USA | Update |
| 3000 | *(Water unit)* | cooling H₂O | Satellite account | Water | USA | Update |

Save the file, then apply it below.

In [ ]:
db.read_add_sectors_excel(ADD_PATH, read_inventories=True)
db.add_sectors(io=ADD_PATH)
db.calc_all(["X", "z", "v", "e", "Y"])

print("Sectors now:", db.sectors)
ai_X = output_series(db, "baseline")
print("\nHyperscalers AI output (M€):", float(ai_X.loc[(AI_REGION, AI_SECTOR)]))
baseline_X = ai_X   # refresh the baseline reference (now 8 sectors)

In [ ]:
# Sanity-check the column we just created: AI's technical-coefficient column
db.z.loc[:, (AI_REGION, "Sector", AI_SECTOR)].pipe(lambda s: s[s != 0])

## Part C — Scenario: EU services swap high-skill labour for AI tokens

The headline question. We assume European **Services** firms reroute a slice of
their spending: instead of paying for **high-skill labour** (a value-added
component), they **buy AI token services from the USA**.

We implement this as a balanced shift in the *Services / Europe* cost column:

- **subtract** an amount `Δ` from the labour value-added coefficient;
- **add** the same `Δ` as an intermediate purchase from *Hyperscalers AI (USA)*.

Because we move `Δ` from value added to intermediate inputs, the column still sums
to 1 — the economy stays balanced. We size `Δ` as a fraction of the current
labour coefficient.

In [ ]:
# Reference values for filling the shock. The labour factor was identified in B.2.
labour_coeff = float(db.v.xs(LABOUR_LABEL)[("Europe", "Sector", "Services")])
SUBSTITUTION_SHARE = 0.15                      # reroute 15% of high-skill labour spend
delta = labour_coeff * SUBSTITUTION_SHARE
print(f"Labour factor             : {LABOUR_LABEL}")
print(f"EU Services labour coeff. : {labour_coeff:.4f}")
print(f"Reroute to AI (delta)     : {delta:.4f}")
print(f"New labour coeff. to write : {labour_coeff - delta:.4f}   (= labour - delta)")

In [ ]:
SUB_PATH = os.path.join(OUT_DIR, "shock_ai_substitution.xlsx")
db.get_shock_excel(path=SUB_PATH, num_shock=2)
print("Empty shock template written to:", SUB_PATH)

Open `outputs/shock_ai_substitution.xlsx` and fill **two** sheets, using the
numbers printed above (`type = Update` sets the coefficient to the given value):

**`v` sheet** — cut the labour coefficient of EU Services to `labour − delta`:

| Factor of production_from | Region_to | Sector_to | type | value |
|---|---|---|---|---|
| *(your labour factor)* | Europe | Services | Update | *(new labour coeff., printed above)* |

**`z` sheet** — add an AI-tokens input into EU Services equal to `delta`:

| Region_from | Sector_from | Region_to | Sector_to | type | value |
|---|---|---|---|---|---|
| USA | Hyperscalers AI | Europe | Services | Update | *(delta, printed above)* |

Moving `delta` from value added to intermediate inputs keeps the column summed to
1 — the economy stays balanced. Save, then apply below.

In [ ]:
db.shock_calc(SUB_PATH, z=True, v=True, scenario="ai_substitution")
print("Scenarios:", db.scenarios)

## Part D — Winners and losers

Now we read the scenario. The substitution does two things at once:

- it **creates demand** for AI compute (USA) and everything that feeds it
  (electrical equipment, electricity, rare materials) — the **winners**;
- it **shrinks** high-skill labour income in Europe and can ripple into the
  sectors that depended on it — the **losers**.

We compare `ai_substitution` against `baseline` along three dimensions: sectoral
output, regional GDP, and CO₂.

In [ ]:
sub_X = output_series(db, "ai_substitution")

comp = pd.DataFrame({"baseline": baseline_X, "ai_substitution": sub_X})
comp["delta"] = comp["ai_substitution"] - comp["baseline"]
comp["pct"] = comp["delta"] / comp["baseline"].replace(0, np.nan) * 100

# Output change by sector (summed across regions)
by_sector = comp.groupby(level="Item")[["baseline", "ai_substitution"]].sum()
by_sector["pct"] = (by_sector["ai_substitution"] / by_sector["baseline"] - 1) * 100
by_sector.sort_values("pct", ascending=False)

In [ ]:
# Winners & losers — sectoral output % change
sdf = by_sector.reset_index().rename(columns={"Item": "Sector"})
sdf["effect"] = np.where(sdf["pct"] >= 0, "winner", "loser")
fig = db.plot(
    data=sdf.sort_values("pct"),
    kind="bar",
    preset=None,
    x="Sector",
    y="pct",
    color="effect",
    title="Winners & losers: sectoral output change (AI substitution vs baseline)",
    path=False,
    auto_open=False,
)
fig

In [ ]:
# Where do the AI-driven gains land geographically? Output change by region.
by_region = comp.groupby(level="Region")[["baseline", "ai_substitution"]].sum()
by_region["pct"] = (by_region["ai_substitution"] / by_region["baseline"] - 1) * 100
rdf = by_region.reset_index()
fig = db.plot(
    data=rdf,
    kind="bar",
    preset=None,
    x="Region",
    y="pct",
    title="Output change by macro-region (AI substitution vs baseline)",
    path=False,
    auto_open=False,
)
fig

In [ ]:
# GDP comparison by region (collapse whatever index GDP returns down to Region)
def gdp_by_region(database, scenario):
    g = database.GDP(scenario=scenario, total=False)
    s = g.iloc[:, 0]
    return s.groupby(level="Region").sum()

gdp_cmp = pd.DataFrame({
    "baseline": gdp_by_region(db, "baseline"),
    "ai_substitution": gdp_by_region(db, "ai_substitution"),
})
gdp_cmp["pct"] = (gdp_cmp["ai_substitution"] / gdp_cmp["baseline"] - 1) * 100
gdp_cmp

In [ ]:
# CO2 by region: does shifting compute to the (electricity-hungry) USA move
# emissions around the map?
def co2_by_region(database, scenario):
    E = database.query("E", scenario)      # absolute satellite-account flows
    co2 = E.loc["CO2"]                      # Series over (Region, Level, Item) columns
    return co2.groupby(level="Region").sum()

try:
    co2_cmp = pd.DataFrame({
        "baseline": co2_by_region(db, "baseline"),
        "ai_substitution": co2_by_region(db, "ai_substitution"),
    })
    co2_cmp["delta"] = co2_cmp["ai_substitution"] - co2_cmp["baseline"]
    display(co2_cmp)
except Exception as exc:
    print("Adjust the CO2 slicing to your exact index layout:", exc)

### Reading the result

Typical story this scenario tells (your exact numbers depend on the calibration):

- **Winners:** *Hyperscalers AI* (USA, by construction), and its supply chain —
  *Electrical equipment*, *Non-renewable* & *Renewable electricity*, *Rare
  materials*. The gains concentrate in the **USA** and in **Rest of Asia / China**
  (chips and metals).
- **Losers:** **European high-skill labour income** falls (value added rerouted to
  imported AI services), and Europe's GDP share slips even if its Services output
  holds up — value that used to be paid to European workers now flows to US
  compute providers.
- **Emissions:** CO₂ tends to **relocate to the USA**, where the electricity- and
  hardware-intensive compute is produced.

### 🧩 Experiment

- Raise `SUBSTITUTION_SHARE` to 0.30 — does Europe's GDP loss scale linearly?
- Make the AI sector greener (shift its electricity mix toward *Renewable
  electricity*) and re-run: how much does the CO₂ relocation shrink?
- Add a second hyperscaler in *Europe* and compare a "sovereign AI" path.

That's the full arc — from raw tables, to a real aggregated model, to a
forward-looking AI scenario. 🎓